# Ablation Study — W1: Joint Hyperparameter Sweep (Wealth Objective)

**Objetivo:** encontrar a combinação de hiperparâmetros que maximiza o retorno financeiro, medido pela *wealth curve* final (`terminal_wealth = Π(1 + r_t)` iniciando em 1.0).

**Dimensões do sweep conjunto (5D):**
- `lambda_penalty` (Jump Penalty λ)
- `vol_estimator` (estimador de volatilidade)
- `n_regimes` (número de regimes)
- `recal_frequency` (frequência de recalibração)
- `forecaster_type` (modelo de previsão de regimes)

## Princípios metodológicos (research-grade)

1. **Ranking primário por `terminal_wealth_val` (janela de validação walk-forward)**, **não** no período de teste completo. Isto evita *hyperparameter leakage on backtest path*.
2. **Reporting honesto**: a config escolhida é reavaliada no **holdout** (`terminal_wealth_oos`) — nunca no mesmo intervalo usado para seleção.
3. **Diagnósticos obrigatórios** (robustez ao *path luck*): `max_drawdown`, `volatility`, `sharpe_ratio`, `turnover`, `n_position_switches`.
4. **Budget computacional explícito**: grid dimensionado para caber em `W1_BUDGET_MAX` combinações.
5. **Interpretabilidade**: decomposição de variância por fator (ANOVA-like) para atribuir quanto de `terminal_wealth` vem de cada dimensão (e não apenas de sorte na trajetória).

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from src.data.loader import DataLoader
from src.config.settings import ASSETS, TEST_START, TEST_END
from src.ablation import (
    ABLATION_W1_CONFIGS,
    W1_BUDGET_MAX,
    W1_VALIDATION_FRACTION,
    run_ablation_sweep,
    analyze_ablation,
    best_config_argmax,
)
from src.ablation.polars_utils import variance_decomposition_polars, float_nan_to_null

print(f'W1 grid size:        {len(ABLATION_W1_CONFIGS)} configs')
print(f'W1 budget:           {W1_BUDGET_MAX}')
print(f'Validation fraction: {W1_VALIDATION_FRACTION:.0%} (walk-forward)')
print(f'Test window:         {TEST_START} → {TEST_END}')

## 1. Execução do Sweep (ou carga a partir do cache)

O sweep W1 é caro (product de 5 dimensões × ativos × seeds). Se já existe um `parquet` em `results/ablation/ablation_W1.parquet`, o notebook o carrega; caso contrário, executa. Para runs ad-hoc, use um subconjunto de ativos em `ASSETS_RUN`.

In [ ]:
RESULTS_DIR = ROOT / 'results' / 'ablation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
W1_PATH = RESULTS_DIR / 'ablation_W1.parquet'

if W1_PATH.exists():
    W1_DF = pl.read_parquet(W1_PATH)
    print(f'Carregado cache: {W1_PATH}  ({W1_DF.height} linhas)')
else:
    # Subset de ativos para demo. Em produção: ASSETS_RUN = list(ASSETS).
    ASSETS_RUN = list(ASSETS)[:3]
    print(f'Executando W1 sweep em {len(ASSETS_RUN)} ativos...')
    loader = DataLoader(cache_dir=str(ROOT / 'data' / 'raw'))
    prices = loader.load_asset_prices()
    er     = loader.excess_returns(prices)
    rf     = loader.risk_free_rate()
    fred   = loader.macro_features()
    W1_DF = run_ablation_sweep(
        ablation_id  = 'W1',
        assets       = ASSETS_RUN,
        er           = er,
        rf           = rf,
        fred         = fred,
        n_bootstrap  = 5,
        n_jobs       = -1,
    )
    W1_DF.write_parquet(W1_PATH)
    print(f'Salvo em {W1_PATH}  ({W1_DF.height} linhas)')

W1_DF = float_nan_to_null(W1_DF)
W1_DF.head(3)

## 2. Seleção walk-forward (sem leakage)

A combinação *vencedora* é escolhida **exclusivamente** pela wealth terminal na janela de validação (`terminal_wealth_val`). O valor em `terminal_wealth` é reservado apenas para reporting final — **nunca** para seleção.

In [ ]:
sel = best_config_argmax(
    W1_DF,
    metric='terminal_wealth_val',
    tiebreak_cols=[('max_drawdown', True), ('turnover', False)],
    higher_is_better=True,
)

print('--- Seleção (validação walk-forward) ---')
print(f"Config escolhida:       {sel['config']}")
print(f"terminal_wealth_val:    {sel['metric_value']:.4f}")
print(f"max_drawdown (mean):    {sel['details'].get('max_drawdown', float('nan')):.4f}")
print(f"turnover (mean):        {sel['details'].get('turnover', float('nan')):.4f}")

# Reporting honesto: wealth na janela de holdout (não usada na seleção)
chosen = (
    W1_DF.filter(pl.col('config') == sel['config'])
         .select(['asset', 'seed', 'terminal_wealth',
                  'terminal_wealth_val', 'terminal_wealth_oos',
                  'max_drawdown', 'volatility', 'sharpe_ratio',
                  'turnover', 'n_position_switches'])
)
print('\n--- Diagnóstico agregado da config escolhida ---')
chosen.select(pl.all().exclude(['asset', 'seed']).mean()).to_pandas().T.round(4)

## 3. Ranking completo por `terminal_wealth_val` com diagnósticos

Mostrar as top-K configs **sempre** com risco/estabilidade ao lado da wealth. O ranking por wealth sem contexto de drawdown é um indicador perigoso (path dependence).

In [ ]:
K = 10
ranking = (
    W1_DF.group_by('config')
         .agg([
             pl.col('terminal_wealth_val').mean().alias('wealth_val'),
             pl.col('terminal_wealth_oos').mean().alias('wealth_oos'),
             pl.col('terminal_wealth').mean().alias('wealth_full'),
             pl.col('max_drawdown').mean().alias('mdd'),
             pl.col('volatility').mean().alias('vol'),
             pl.col('sharpe_ratio').mean().alias('sharpe'),
             pl.col('turnover').mean().alias('turn'),
             pl.col('n_position_switches').mean().alias('n_switches'),
         ])
         .sort('wealth_val', descending=True)
         .head(K)
)
ranking

## 4. Consistência validação ↔ holdout

Se o ranking por `terminal_wealth_val` discordar fortemente do ranking por `terminal_wealth_oos`, a seleção está capturando *path luck* e não estrutura — alerta para banca.

In [ ]:
by_cfg = (
    W1_DF.group_by('config')
         .agg([
             pl.col('terminal_wealth_val').mean().alias('wealth_val'),
             pl.col('terminal_wealth_oos').mean().alias('wealth_oos'),
         ])
         .drop_nulls()
         .to_pandas()
)
rho = by_cfg[['wealth_val', 'wealth_oos']].corr().iloc[0, 1]
rank_rho = by_cfg[['wealth_val', 'wealth_oos']].rank().corr().iloc[0, 1]
print(f'Pearson(val, oos):  {rho:.3f}')
print(f'Spearman rank corr: {rank_rho:.3f}')

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(by_cfg['wealth_val'], by_cfg['wealth_oos'], alpha=0.6)
lim = [min(by_cfg.min()), max(by_cfg.max())]
ax.plot(lim, lim, 'k--', lw=0.8)
ax.set_xlabel('terminal_wealth_val  (selection)')
ax.set_ylabel('terminal_wealth_oos  (holdout)')
ax.set_title(f'Consistência seleção ↔ holdout  ·  Spearman={rank_rho:.2f}')
plt.tight_layout(); plt.show()

## 5. Atribuição de variância (interpretabilidade)

Decomposição ANOVA-like: quanto da variância de `terminal_wealth_val` cada fator isolado explica? Em um sweep conjunto sem decomposição, ganhos podem vir de interações puras — a banca vai perguntar *qual mecanismo moveu a wealth*.

In [ ]:
# Recupera os fatores do nome da config (w1_l{LAM}_{VOL}_k{K}_{RECAL}_{FC}).
def _parse_w1_name(name: str):
    parts = name.split('_')
    return {
        'lambda_penalty':  int(parts[1].lstrip('l')),
        'vol_estimator':   parts[2],
        'n_regimes':       int(parts[3].lstrip('k')),
        'recal_frequency': parts[4],
        'forecaster_type': parts[5],
    }

_factors_pd = pd.DataFrame([
    _parse_w1_name(n) for n in W1_DF['config'].to_list()
])
W1_FACTORS = W1_DF.hstack(pl.from_pandas(_factors_pd))

vd = variance_decomposition_polars(
    W1_FACTORS.drop_nulls(subset=['terminal_wealth_val']),
    factor_cols=['lambda_penalty', 'vol_estimator', 'n_regimes',
                 'recal_frequency', 'forecaster_type'],
    metric_col='terminal_wealth_val',
)
vd

In [ ]:
vd_pd = vd.to_pandas().sort_values('pct_variance')
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(vd_pd['factor'], vd_pd['pct_variance'])
ax.set_xlabel('% variância de terminal_wealth_val (ANOVA-like)')
ax.set_title('W1 — Atribuição de variância por fator')
plt.tight_layout(); plt.show()

## 6. Perfil de risco da config vencedora

Relatório final que a banca espera ver em conjunto: wealth + risco + estabilidade.

In [ ]:
def fmt(v, d=4):
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return 'NaN'
    return f'{v:.{d}f}'

chosen_pd = chosen.to_pandas()
summary = {
    'config (argmax val)':    sel['config'],
    'wealth_val (selection)': fmt(chosen_pd['terminal_wealth_val'].mean()),
    'wealth_oos (holdout)':   fmt(chosen_pd['terminal_wealth_oos'].mean()),
    'wealth_full (report)':   fmt(chosen_pd['terminal_wealth'].mean()),
    'sharpe_ratio':           fmt(chosen_pd['sharpe_ratio'].mean()),
    'max_drawdown':           fmt(chosen_pd['max_drawdown'].mean()),
    'volatility':             fmt(chosen_pd['volatility'].mean()),
    'turnover':               fmt(chosen_pd['turnover'].mean()),
    'n_position_switches':    fmt(chosen_pd['n_position_switches'].mean(), 1),
}
pd.DataFrame(summary, index=['mean']).T

## Notas de defesa

- **Sem leakage**: seleção apenas em `terminal_wealth_val` (primeiros `W1_VALIDATION_FRACTION = 60%` dos retornos do período de teste) — nunca em `terminal_wealth` completo.
- **Tie-break explícito**: em caso de wealth empatada, prioridade para menor drawdown e menor turnover, o que torna a escolha defensável contra *path luck*.
- **Interpretabilidade**: a decomposição de variância indica se o ganho vem de um fator dominante ou de interações. Se for interação pura, documentar explicitamente.
- **Budget**: grid caiu em `W1_BUDGET_MAX = 150` (produto cartesiano de valores já usados em A1/A2/A3/B1/D1), sem amostragem estocástica.